In [ ]:
from keras import layers

def se_block(input_tensor, ratio=8):
    """Squeeze and Excitation block to add attention over feature maps."""
    channel_axis = -1
    filters = input_tensor.shape[channel_axis]
    se_shape = (1, 1, filters)

    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape(se_shape)(se)
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)

    x = layers.multiply([input_tensor, se])
    return x

from keras import backend as K
from keras import layers

import tensorflow as tf
from tensorflow.keras import layers

def spatial_attention(input_tensor):
    """Spatial attention module."""
    # Define the average pooling and max pooling operations
    avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True), 
                             output_shape=lambda s: (s[0], s[1], s[2], 1))(input_tensor)
    
    max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True), 
                             output_shape=lambda s: (s[0], s[1], s[2], 1))(input_tensor)
    
    # Concatenate the outputs of average and max pooling
    concat = layers.concatenate([avg_pool, max_pool], axis=-1)
    
    # Apply convolution to get the attention map
    attention = layers.Conv2D(1, (7, 7), padding='same', activation='sigmoid')(concat)
    
    # Multiply the input tensor by the attention map
    output = layers.multiply([input_tensor, attention])
    
    return output





# Encoder
encoder_input = layers.Input(shape=input_shape)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=regularizers.l2(0.001))(encoder_input)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.BatchNormalization()(x)
# x = se_block(x)  # Add SE block for attention

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.BatchNormalization()(x)
x = spatial_attention(x)  # Add spatial attention here

encoded = layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=regularizers.l1(0.001))(x)
encoded = layers.Dropout(0.5)(encoded)

# Decoder
x = layers.UpSampling2D((2, 2))(encoded)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
decoded = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

# Autoencoder model
autoencoder = models.Model(encoder_input, decoded)
autoencoder.compile(optimizer='adam', loss='mse')


In [ ]:
from keras.models import Model
import numpy as np
import matplotlib.pyplot as plt

# Choose the layer to inspect (e.g., first conv layer)
layer_name = 'conv2d_6'  # You can inspect any of the conv layers by name
layer_output = autoencoder.get_layer(layer_name).output

# Create a new model that will return the output of this layer given the input
activation_model = Model(inputs=autoencoder.input, outputs=layer_output)

# Choose an input sample for visualization (e.g., the first sample in the validation set)
input_image = np.expand_dims(sdi_0[0], axis=0)  # Add batch dimension if needed

# Get the activations for the chosen layer
activations = activation_model.predict(input_image)

# Plot the feature maps for each filter in the chosen layer
def display_feature_maps(activations, layer_index):
    num_filters = activations.shape[-1]
    size = activations.shape[1]
    n_cols = 8
    n_rows = num_filters // n_cols
    
    plt.figure(figsize=(n_cols, n_rows))
    for i in range(num_filters):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(activations[0, :, :, i], cmap='viridis')  # Plot the feature map
        plt.axis('off')
    plt.show()

# Visualize feature maps of the chosen layer
display_feature_maps(activations, 0)
